# 🛡️ EdgeShield: Small Language Model (SLM) Training Pipeline on Kaggle
**Doctoral Research Milestone | IEEE TEMSCON-ASPAC 2026 Accepted Paper**  
*Paper Title:* EdgeShield: A Small Language Model Framework for Real-Time Detection of Lateral Movement and Ransomware Attacks  
*Authors:* Gauravkumar Raval (Research Scholar) & Dr. Tejaskumar Bhatt (Supervisor)  
*Affiliation:* Faculty of Engineering & Technology, GLS University, Ahmedabad, India  

---

### ⚡ Why Kaggle for EdgeShield SLM Training?
- **30 Hours/Week Free GPU**: NVIDIA Tesla T4 x2 (dual GPU) or NVIDIA Tesla P100 (16 GB VRAM).
- **30 GB System RAM**: Eliminates memory pressure and pandas OOM when loading the 1.07 GB authentic LMD-2023 dataset.
- **73 GB Disk Space**: Ample room for base SLM checkpoints, ONNX quantized models, and raw Sysmon telemetry.
- **12-Hour Continuous Session Timeout**: Long-running training jobs won't get prematurely disconnected.

> ⚠️ **IMPORTANT KAGGLE SETTING**:  
> In the right-hand panel under **Notebook options**:
> 1. **Accelerator**: Select `GPU T4 x 2` (or `GPU P100`).
> 2. **Internet**: Toggle to **`Internet on`** (required for `git clone`, `pip`, `gdown`, and HuggingFace weights).

## 1. ⚙️ Hardware & GPU Verification
Verify NVIDIA CUDA acceleration, VRAM allocation, and system memory.

In [ ]:
!nvidia-smi

import torch
import psutil

ram_gb = psutil.virtual_memory().total / (1024 ** 3)
print("=" * 60)
print(f"System RAM:       {ram_gb:.1f} GB")
print(f"PyTorch Version:  {torch.__version__}")
print(f"CUDA Available:   {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Primary GPU:      {torch.cuda.get_device_name(0)}")
    print(f"GPU Count:        {torch.cuda.device_count()}")
    print(f"Allocated VRAM:   {torch.cuda.memory_allocated(0)/(1024**2):.2f} MB")
    print("[+] Hardware acceleration is ACTIVE and ready for training!")
else:
    print("[!] WARNING: GPU accelerator not active. In right panel -> Accelerator -> Select GPU T4 x2.")
print("=" * 60)

## 2. 📂 Clone Repository & Enter Working Directory
Clones `slm-gpu-trainer` into `/kaggle/working/slm-gpu-trainer` on branch `feat/edgeshield-slm-framework`.

In [ ]:
import os

REPO_DIR = "/kaggle/working/slm-gpu-trainer"
BRANCH = "feat/edgeshield-slm-framework"

if not os.path.exists(REPO_DIR):
    print(f"[*] Cloning repository on branch {BRANCH}...")
    !git clone -b {BRANCH} https://github.com/graval/slm-gpu-trainer.git {REPO_DIR}
else:
    print("[*] Updating repository...")
    !cd {REPO_DIR} && git pull origin {BRANCH}

%cd /kaggle/working/slm-gpu-trainer/slm-tr
!mkdir -p data models models/onnx_edgeshield
!pwd

## 3. 📦 Install Dependencies
Kaggle environments come with PyTorch, CUDA, transformers, and scikit-learn preinstalled. We install complementary packages for ONNX export, INT8 quantization, and RAR decompression.

In [ ]:
# Install specialized quantization, PEFT, and utilities
!pip install -q onnx onnxruntime onnxscript peft>=0.9.0 trl>=0.8.0 bitsandbytes gdown
!apt-get update -qq && apt-get install -y -qq unrar
print("[+] Dependencies initialized successfully!")

## 4. 📊 Dataset Setup: Authentic LMD-2023 Lateral Movement Dataset
Acquires the real, peer-reviewed **LMD-2023** dataset from the author repository via `gdown`, uncompresses the 1.75M element labeled Sysmon telemetry archive (~1.07 GB), and generates OpTC and ransomware benchmark suites.

In [ ]:
import os

# Check if dataset is mounted via Kaggle Datasets, otherwise download via gdown
kaggle_input_candidates = [
    "/kaggle/input/lateral-movement-dataset-lmd-2023/lmd_2023_dataset.csv",
    "/kaggle/input/lmd-2023/lmd_2023_dataset.csv",
    "/kaggle/input/lmd2023/lmd_2023_dataset.csv"
]

dataset_found = False
for cand in kaggle_input_candidates:
    if os.path.exists(cand):
        !cp "{cand}" data/lmd_2023_dataset.csv
        print(f"[+] Dataset attached directly from Kaggle Input: {cand}")
        dataset_found = True
        break

if not dataset_found and not os.path.exists("data/lmd_2023_dataset.csv"):
    print("[*] Downloading official author LMD-2023 dataset from Google Drive...")
    !gdown --folder "https://drive.google.com/drive/folders/1PkJiGpD0Kn1rV8GC9m2b_eWvqQTSa1eH?usp=sharing" -O data/
    
    # Extract archives
    if os.path.exists("data/Lateral-Movement-Dataset--LMD_Collections/LMD-2023/LMD-2023.rar"):
        !unrar x -o+ "data/Lateral-Movement-Dataset--LMD_Collections/LMD-2023/LMD-2023.rar" data/
    if os.path.exists("data/LMD-2023 [1.75M Elements].rar"):
        !unrar x -o+ "data/LMD-2023 [1.75M Elements].rar" data/
    
    # Activate authentic 1.75M Sysmon CSV
    extracted_csv = "data/LMD-2023 [1.75M Elements]/LMD-2023 [1.75M Elements] Checked/Labelled LMD-2023/LMD-2023 [1.75M Elements][Labelled]checked.csv"
    if os.path.exists(extracted_csv):
        !cp "{extracted_csv}" data/lmd_2023_dataset.csv
        print("[+] Activated 1.75M Sysmon telemetry dataset (~1.07 GB)!")
    elif os.path.exists("data/Lateral-Movement-Dataset--LMD_Collections/full-csv(Evaluated Titles).csv"):
        !cp "data/Lateral-Movement-Dataset--LMD_Collections/full-csv(Evaluated Titles).csv" data/lmd_2023_dataset.csv
        print("[+] Fallback: Activated full-csv(Evaluated Titles).csv")

# Generate curated DARPA OpTC and ransomware evaluation sets
from edgeshield.data.synthetic_ransomware import generate_curated_ransomware_dataset
generate_curated_ransomware_dataset("data/ransomware_behavioral_traces.csv", num_samples=12000)

from scripts.prepare_optc_benchmark import generate_optc_test_benchmark
generate_optc_test_benchmark("data/optc_test_benchmark.csv", total_samples=1500)

!ls -lh data/

## 5A. 🎯 Train Stream A: Log Semantic Analyzer (LSA)
Trains the DeBERTa-v3-small SLM backbone for Windows authentication & Lateral Movement detection (MITRE ATT&CK TA0008) with Supervised Cross-Entropy and Contrastive Loss.

In [ ]:
!python scripts/train_edgeshield_lsa.py \
    --model_name "microsoft/deberta-v3-small" \
    --dataset_path "data/lmd_2023_dataset.csv" \
    --device "cuda" \
    --epochs 3 \
    --batch_size 16 \
    --lr 5e-5 \
    --use_contrastive \
    --num_samples 12000 \
    --test_samples 2000 \
    --output_dir "models/edgeshield_lsa_deberta"

## 5B. ⚡ Train Stream B: Behavioral Pattern Detector (BPD)
Fine-tunes the SLM sequence classifier and **Attention Threat Scoring Head** on ransomware behavioral telemetry (T1490, T1562.001, T1486) for pre-encryption intercept.

In [ ]:
!python scripts/train_edgeshield_bpd.py \
    --model_name "distilbert-base-uncased" \
    --dataset_path "data/ransomware_behavioral_traces.csv" \
    --device "cuda" \
    --epochs 3 \
    --batch_size 16 \
    --threat_weight 0.5 \
    --num_samples 8000 \
    --test_samples 1500 \
    --output_dir "models/edgeshield_bpd"

## 5C. 🔄 (Optional) Train Sequence Classifier (v2 Sliding Window, K=3)
Trains the multi-class lateral movement classifier across sliding temporal windows ($K=3$) using Hugging Face Trainer on CUDA GPU.

In [ ]:
!python train_classifier.py \
    --variant v2 \
    --window_size 3 \
    --device cuda \
    --epochs 3 \
    --batch_size 16 \
    --model_name "microsoft/deberta-v3-small" \
    --csv_path "data/lmd_2023_dataset.csv" \
    --output_dir "models/deberta-lateral-movement-v2_sliding_window"

## 5D. 🧠 (Optional) Train Tier 2 Generative Reasoner (LoRA / QLoRA)
Fine-tunes `Qwen/Qwen2.5-1.5B-Instruct` with Parameter-Efficient Fine-Tuning (PEFT LoRA) and 4-bit quantization (QLoRA) to generate causal JSON incident reports.

In [ ]:
import os
os.environ["PYTHONUTF8"] = "1"

!python train_generator.py \
    --model_name "Qwen/Qwen2.5-1.5B-Instruct" \
    --variant v2 \
    --window_size 3 \
    --csv_path "data/lmd_2023_dataset.csv" \
    --num_samples 4000 \
    --epochs 1 \
    --batch_size 2 \
    --gradient_accumulation_steps 4 \
    --learning_rate 2e-4 \
    --qlora \
    --output_dir "models/qwen-lateral-movement-v2_sliding_window"

## 6. 🚀 ONNX Export & Dynamic INT8 Quantization
Exports PyTorch weights to ONNX format and quantizes dynamically to INT8 precision (achieving 11.71 ms latency and <50MB RAM footprint on edge hardware).

In [ ]:
# Export PyTorch model to ONNX and dynamically quantize to INT8
!python scripts/export_onnx.py \
    --checkpoint "models/edgeshield_lsa_deberta" \
    --out-dir "models/onnx_edgeshield" \
    --name "edgeshield_lsa" \
    --quantize

# Benchmark ONNX INT8 latency across 500 runs
!python scripts/benchmark_onnx.py \
    --model "models/onnx_edgeshield/edgeshield_lsa_int8.onnx" \
    --runs 500 \
    --export_summary

## 7. 📈 Empirical Benchmark & Multi-Stage Attack Scenarios
Validates detection against baseline classifiers (XGBoost, Random Forest) across multi-stage attack scenarios.

In [ ]:
!python scripts/benchmark_edgeshield.py \
    --lsa_model "models/edgeshield_lsa_deberta" \
    --bpd_model "models/edgeshield_bpd" \
    --optc_path "data/optc_test_benchmark.csv" \
    --ransomware_path "data/ransomware_behavioral_traces.csv" \
    --lmd_path "data/lmd_2023_dataset.csv" \
    --num_samples 500 \
    --device cuda \
    --export_json "benchmark_results.json"

## 8. 💾 Kaggle Output Persistence
All artifacts placed in `/kaggle/working/` persist into the notebook's **Output** tab upon completion. We archive all trained models and ONNX binaries into a single portable zip archive.

In [ ]:
import shutil

# Archive models and benchmarks into a downloadable zip file in /kaggle/working
print("[*] Archiving trained models and ONNX bundles into /kaggle/working/edgeshield_models.zip...")
!zip -r /kaggle/working/edgeshield_trained_models.zip models/ benchmark_results.json

!ls -lh /kaggle/working/edgeshield_trained_models.zip
print("[+] Ready! You can download edgeshield_trained_models.zip directly from the Kaggle Output tab on the right panel.")